In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.model_selection import StratifiedKFold


CURRENT_DIR = Path.cwd().resolve()

PROJECT_ROOT = next(
    (
        path
        for path in [CURRENT_DIR, *CURRENT_DIR.parents]
        if (path / "src" / "feature_engineering.py").exists()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f"Не найден корень проекта. Текущая папка: {CURRENT_DIR}"
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.feature_engineering import (
    prepare_features,
    add_title_hierarchy_features,
)


PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports"

TARGET_COLUMN = "Цена"

train = pd.read_parquet(
    PROCESSED_DIR / "train_canonical.parquet"
)

X_raw = train.drop(columns=[TARGET_COLUMN]).copy()
y = train[TARGET_COLUMN].copy()

X_features = add_title_hierarchy_features(
    prepare_features(X_raw)
)

In [2]:
from pathlib import Path
import sys
import importlib

import numpy as np
import pandas as pd

from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.model_selection import (
    StratifiedKFold,
    train_test_split,
)

CURRENT_DIR = Path.cwd().resolve()

PROJECT_ROOT = next(
    (
        path
        for path in [CURRENT_DIR, *CURRENT_DIR.parents]
        if (path / "src" / "feature_engineering.py").exists()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f"Не найден корень проекта. Current directory: {CURRENT_DIR}"
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import src.feature_engineering as fe
importlib.reload(fe)

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

train = pd.read_parquet(
    PROCESSED_DIR / "train_canonical.parquet"
)

TARGET_COLUMN = "Цена"

X_raw = train.drop(columns=[TARGET_COLUMN]).copy()
y = train[TARGET_COLUMN].copy()

X_features = fe.add_title_hierarchy_features(
    fe.prepare_features(X_raw)
)

RAW_DUPLICATE_NUMERIC_COLUMNS = [
    "Пробег",
    "Расход",
    "Количество цилиндров",
    "Двери",
    "Количество кресел",
]

EXCLUDED_COLUMNS = [
    "car_id",
    "Предложение",
] + RAW_DUPLICATE_NUMERIC_COLUMNS

EXCLUDED_COLUMNS_V7 = EXCLUDED_COLUMNS + [
    "Полное название",
    "Цвет",
]

feature_columns_v7 = [
    column
    for column in X_features.columns
    if column not in EXCLUDED_COLUMNS_V7
]

X_model_v7 = X_features[feature_columns_v7].copy()

numeric_columns_v7 = X_model_v7.select_dtypes(
    include=["number", "bool"]
).columns.tolist()

categorical_columns_v7 = [
    column
    for column in feature_columns_v7
    if column not in numeric_columns_v7
]

for column in numeric_columns_v7:
    X_model_v7[column] = pd.to_numeric(
        X_model_v7[column],
        errors="coerce",
    ).astype(float)

for column in categorical_columns_v7:
    X_model_v7[column] = (
        X_model_v7[column]
        .astype("string")
        .fillna("__MISSING__")
        .astype(str)
    )

assert X_model_v7.shape[1] == 57
assert "Полное название" not in X_model_v7.columns
assert "Цвет" not in X_model_v7.columns

price_bins = np.asarray(
    pd.qcut(
        y,
        q=10,
        labels=False,
        duplicates="drop",
    )
)

outer_train_idx, outer_valid_idx = train_test_split(
    np.arange(len(y)),
    test_size=0.20,
    random_state=42,
    stratify=price_bins,
)

X_train_base = X_model_v7.iloc[outer_train_idx].copy()
X_valid_base = X_model_v7.iloc[outer_valid_idx].copy()

y_train_outer = y.iloc[outer_train_idx].copy()
y_valid_outer = y.iloc[outer_valid_idx].copy()

print("v7 train:", X_train_base.shape)
print("v7 valid:", X_valid_base.shape)

v7 train: (6672, 57)
v7 valid: (1668, 57)


Функции для leakage-safe статистик

In [3]:
def make_group_key(
    df: pd.DataFrame,
    columns: list[str],
) -> pd.Series:
    key = None

    for column in columns:
        series = df[column]

        if pd.api.types.is_numeric_dtype(series):
            piece = (
                pd.to_numeric(
                    series,
                    errors="coerce",
                )
                .round(4)
                .astype("Float64")
                .astype("string")
            )
        else:
            piece = series.astype("string")

        piece = (
            piece
            .fillna("__MISSING__")
            .str.strip()
            .str.upper()
        )

        key = piece if key is None else key.str.cat(
            piece,
            sep="|||",
        )

    return key


def fit_group_mapping(
    keys: pd.Series,
    y_log: np.ndarray,
    smoothing: float,
):
    global_mean = float(np.mean(y_log))

    group_frame = pd.DataFrame(
        {
            "key": keys.to_numpy(dtype=object),
            "target_log": np.asarray(y_log, dtype=float),
        }
    )

    stats = (
        group_frame
        .groupby("key", dropna=False, sort=False)
        .agg(
            count=("target_log", "size"),
            target_sum=("target_log", "sum"),
        )
    )

    stats["smooth_mean_log_price"] = (
        stats["target_sum"] + smoothing * global_mean
    ) / (
        stats["count"] + smoothing
    )

    return (
        stats[
            [
                "count",
                "smooth_mean_log_price",
            ]
        ],
        global_mean,
    )


def transform_group_mapping(
    keys: pd.Series,
    mapping: pd.DataFrame,
    global_mean: float,
):
    count = (
        pd.to_numeric(
            keys.map(mapping["count"]),
            errors="coerce",
        )
        .fillna(0)
        .to_numpy(dtype=float)
    )

    smooth_mean = (
        pd.to_numeric(
            keys.map(mapping["smooth_mean_log_price"]),
            errors="coerce",
        )
        .fillna(global_mean)
        .to_numpy(dtype=float)
    )

    return (
        np.log1p(count),
        smooth_mean,
    )


def make_cross_fitted_target_stats(
    X_fit: pd.DataFrame,
    y_fit: pd.Series,
    X_apply: pd.DataFrame,
    group_specs: dict[str, list[str]],
    smoothing: float = 20.0,
    n_splits: int = 5,
    random_state: int = 142,
):
    y_fit_array = y_fit.to_numpy(dtype=float)
    y_fit_log = np.log1p(y_fit_array)

    inner_bins = np.asarray(
        pd.qcut(
            y_fit_array,
            q=10,
            labels=False,
            duplicates="drop",
        )
    )

    inner_cv = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state,
    )

    fit_stats = pd.DataFrame(index=X_fit.index)
    apply_stats = pd.DataFrame(index=X_apply.index)

    for group_name, group_columns in group_specs.items():
        fit_keys = make_group_key(
            X_fit,
            group_columns,
        ).reset_index(drop=True)

        apply_keys = make_group_key(
            X_apply,
            group_columns,
        ).reset_index(drop=True)

        fit_log_count = np.zeros(len(X_fit), dtype=float)
        fit_smooth_mean = np.zeros(len(X_fit), dtype=float)

        for inner_train_pos, inner_valid_pos in inner_cv.split(
            np.arange(len(X_fit)),
            inner_bins,
        ):
            inner_mapping, inner_global_mean = fit_group_mapping(
                keys=fit_keys.iloc[inner_train_pos],
                y_log=y_fit_log[inner_train_pos],
                smoothing=smoothing,
            )

            inner_log_count, inner_smooth_mean = (
                transform_group_mapping(
                    keys=fit_keys.iloc[inner_valid_pos],
                    mapping=inner_mapping,
                    global_mean=inner_global_mean,
                )
            )

            fit_log_count[inner_valid_pos] = inner_log_count
            fit_smooth_mean[inner_valid_pos] = inner_smooth_mean

        full_mapping, full_global_mean = fit_group_mapping(
            keys=fit_keys,
            y_log=y_fit_log,
            smoothing=smoothing,
        )

        apply_log_count, apply_smooth_mean = (
            transform_group_mapping(
                keys=apply_keys,
                mapping=full_mapping,
                global_mean=full_global_mean,
            )
        )

        fit_stats[
            f"te_{group_name}_log_count"
        ] = fit_log_count

        fit_stats[
            f"te_{group_name}_smooth_mean_log_price"
        ] = fit_smooth_mean

        apply_stats[
            f"te_{group_name}_log_count"
        ] = apply_log_count

        apply_stats[
            f"te_{group_name}_smooth_mean_log_price"
        ] = apply_smooth_mean

    return fit_stats, apply_stats

Построить 12 новых признаков

Пока не добавляем median, std, десятки комбинаций и не перебираем smoothing. Первый тест должен быть чистым.

In [4]:
GROUP_SPECS = {
    "brand_model": [
        "Бренд",
        "Модель",
    ],
    "brand_model_year": [
        "Бренд",
        "Модель",
        "Год выпуска",
    ],
    "title_prefix3": [
        "Название_префикс_3",
    ],
    "title_prefix3_year": [
        "Название_префикс_3",
        "Год выпуска",
    ],
    "title_normalized": [
        "Название_нормализованное_без_года",
    ],
    "title_normalized_year": [
        "Название_нормализованное_без_года",
        "Год выпуска",
    ],
}

X_train_stats, X_valid_stats = make_cross_fitted_target_stats(
    X_fit=X_train_base,
    y_fit=y_train_outer,
    X_apply=X_valid_base,
    group_specs=GROUP_SPECS,
    smoothing=20.0,
)

assert X_train_stats.shape[1] == 12
assert X_valid_stats.shape[1] == 12
assert not X_train_stats.isna().any().any()
assert not X_valid_stats.isna().any().any()

stats_coverage_rows = []

for group_name in GROUP_SPECS:
    count_column = f"te_{group_name}_log_count"

    valid_counts = np.expm1(
        X_valid_stats[count_column].to_numpy()
    )

    matched_mask = valid_counts > 0

    stats_coverage_rows.append(
        {
            "group": group_name,
            "valid_coverage_pct": matched_mask.mean() * 100,
            "median_train_group_size_when_matched": (
                np.median(valid_counts[matched_mask])
                if matched_mask.any()
                else 0
            ),
        }
    )

stats_coverage = pd.DataFrame(stats_coverage_rows)

display(stats_coverage)

X_train_with_stats = pd.concat(
    [
        X_train_base,
        X_train_stats,
    ],
    axis=1,
)

X_valid_with_stats = pd.concat(
    [
        X_valid_base,
        X_valid_stats,
    ],
    axis=1,
)

print("Base shape:", X_train_base.shape)
print("With target stats:", X_train_with_stats.shape)

,group,valid_coverage_pct,median_train_group_size_when_matched
0,brand_model,96.582734,44.0
1,brand_model_year,81.954436,5.0
2,title_prefix3,89.688249,14.0
3,title_prefix3_year,67.146283,3.0
4,title_normalized,76.618705,5.0
5,title_normalized_year,48.561151,2.0


Base shape: (6672, 57)
With target stats: (6672, 69)


Обучить baseline v7 и v7 + target stats

In [5]:
def mape_percent(y_true, y_pred) -> float:
    return mean_absolute_percentage_error(
        y_true,
        y_pred,
    ) * 100


def train_catboost_holdout(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    X_valid: pd.DataFrame,
    y_valid: pd.Series,
    categorical_columns: list[str],
):
    model = CatBoostRegressor(
        loss_function="RMSE",
        iterations=3000,
        learning_rate=0.05,
        depth=8,
        l2_leaf_reg=5,
        random_seed=42,
        verbose=500,
        allow_writing_files=False,
    )

    model.fit(
        X_train,
        np.log1p(y_train),
        cat_features=categorical_columns,
        eval_set=(
            X_valid,
            np.log1p(y_valid),
        ),
        use_best_model=True,
        early_stopping_rounds=200,
    )

    prediction = np.maximum(
        np.expm1(
            model.predict(X_valid)
        ),
        1,
    )

    return model, prediction


def normalize_title_for_shift(
    series: pd.Series,
) -> pd.Series:
    return (
        series
        .astype("string")
        .fillna("__MISSING__")
        .str.upper()
        .str.strip()
    )


title_train = normalize_title_for_shift(
    X_raw.iloc[outer_train_idx]["Полное название"]
)

title_valid = normalize_title_for_shift(
    X_raw.iloc[outer_valid_idx]["Полное название"]
)

valid_title_unseen_mask = (
    ~title_valid.isin(set(title_train))
).to_numpy()

low_price_threshold = y_train_outer.quantile(0.25)

valid_low_price_mask = (
    y_valid_outer.to_numpy() <= low_price_threshold
)

baseline_model_v7, baseline_pred_v7 = train_catboost_holdout(
    X_train=X_train_base,
    y_train=y_train_outer,
    X_valid=X_valid_base,
    y_valid=y_valid_outer,
    categorical_columns=categorical_columns_v7,
)

stats_model_v7, stats_pred_v7 = train_catboost_holdout(
    X_train=X_train_with_stats,
    y_train=y_train_outer,
    X_valid=X_valid_with_stats,
    y_valid=y_valid_outer,
    categorical_columns=categorical_columns_v7,
)

0:	learn: 0.6525543	test: 0.6505617	best: 0.6505617 (0)	total: 238ms	remaining: 11m 53s
500:	learn: 0.1551241	test: 0.2024758	best: 0.2024758 (500)	total: 1m	remaining: 4m 59s
1000:	learn: 0.1189818	test: 0.1929087	best: 0.1928878 (999)	total: 2m 2s	remaining: 4m 4s
1500:	learn: 0.0978764	test: 0.1897706	best: 0.1897523 (1499)	total: 3m 5s	remaining: 3m 5s
2000:	learn: 0.0812090	test: 0.1881756	best: 0.1881697 (1984)	total: 4m 27s	remaining: 2m 13s
2500:	learn: 0.0687433	test: 0.1871906	best: 0.1871594 (2494)	total: 5m 30s	remaining: 1m 5s
2999:	learn: 0.0590468	test: 0.1868294	best: 0.1867765 (2968)	total: 6m 44s	remaining: 0us

bestTest = 0.186776484
bestIteration = 2968

Shrink model to first 2969 iterations.
0:	learn: 0.6523671	test: 0.6494952	best: 0.6494952 (0)	total: 118ms	remaining: 5m 52s
500:	learn: 0.1425179	test: 0.2006706	best: 0.2006692 (499)	total: 52.5s	remaining: 4m 21s
1000:	learn: 0.1045746	test: 0.1912354	best: 0.1912354 (1000)	total: 1m 56s	remaining: 3m 52s
1500:	

In [6]:
comparison_rows = []

for model_name, prediction, model in [
    (
        "v7_baseline",
        baseline_pred_v7,
        baseline_model_v7,
    ),
    (
        "v7_plus_hierarchical_target_stats",
        stats_pred_v7,
        stats_model_v7,
    ),
]:
    comparison_rows.append(
        {
            "model": model_name,
            "best_iteration": model.get_best_iteration(),
            "overall_mape_pct": mape_percent(
                y_valid_outer,
                prediction,
            ),
            "unseen_title_mape_pct": mape_percent(
                y_valid_outer.to_numpy()[
                    valid_title_unseen_mask
                ],
                prediction[
                    valid_title_unseen_mask
                ],
            ),
            "low_price_mape_pct": mape_percent(
                y_valid_outer.to_numpy()[
                    valid_low_price_mask
                ],
                prediction[
                    valid_low_price_mask
                ],
            ),
        }
    )

target_stats_comparison = pd.DataFrame(
    comparison_rows
)

target_stats_comparison["delta_vs_baseline_pp"] = (
    target_stats_comparison["overall_mape_pct"]
    - target_stats_comparison.loc[
        target_stats_comparison["model"].eq(
            "v7_baseline"
        ),
        "overall_mape_pct",
    ].iloc[0]
)

display(target_stats_comparison)

,model,best_iteration,overall_mape_pct,unseen_title_mape_pct,low_price_mape_pct,delta_vs_baseline_pp
0,v7_baseline,2968,12.806429,16.547210,18.769882,0.000000
1,v7_plus_hierarchical_target_stats,2991,12.868201,16.451899,18.700734,0.061772


Базовый v7 воспроизвёлся идеально — это главное:

v7 baseline:                  12.806429%
v7 + hierarchical target stats: 12.868201%
Δ:                            +0.061772 п.п.

То есть основной эксперимент не прошёл. Не расширяем его ни median/std, ни десятками новых групп, ни подбором smoothing: базовый сигнал уже отрицательный.

При этом есть интересная деталь:

unseen title: 16.547 → 16.452
low price:    18.770 → 18.701

Статистики немного помогают сложным сегментам, но вредят обычным объектам сильнее. Это похоже на шумный, но потенциально диверсифицирующий прогноз.

Сделаем один дешёвый контрольный тест: может ли смесь baseline и stats-модели быть лучше baseline. Если нет — ветку закрываем окончательно.

In [7]:
blend_rows = []

for stats_weight in np.arange(0.05, 0.51, 0.05):
    blended_pred = (
        (1 - stats_weight) * baseline_pred_v7
        + stats_weight * stats_pred_v7
    )

    blend_rows.append(
        {
            "stats_weight": stats_weight,
            "baseline_weight": 1 - stats_weight,
            "blend_mape_pct": mape_percent(
                y_valid_outer,
                blended_pred,
            ),
            "delta_vs_baseline_pp": (
                mape_percent(y_valid_outer, blended_pred)
                - mape_percent(
                    y_valid_outer,
                    baseline_pred_v7,
                )
            ),
        }
    )

stats_blend_results = (
    pd.DataFrame(blend_rows)
    .sort_values("blend_mape_pct")
    .reset_index(drop=True)
)

display(stats_blend_results)

,stats_weight,baseline_weight,blend_mape_pct,delta_vs_baseline_pp
0,0.45,0.55,12.566674,-0.239755
1,0.50,0.50,12.567937,-0.238492
2,0.40,0.60,12.570543,-0.235886
3,0.35,0.65,12.580288,-0.226142
4,0.30,0.70,12.594856,-0.211573
5,0.25,0.75,12.613698,-0.192731
6,0.20,0.80,12.638953,-0.167476
7,0.15,0.85,12.669505,-0.136924
8,0.10,0.90,12.707738,-0.098691
9,0.05,0.95,12.753461,-0.052968


Это уже не «немного диверсификации»: модель сама хуже на +0.062, но её смесь с v7 даёт −0.240 п.п. MAPE. Значит, target-stats модель делает другие, полезные ошибки — именно такой сценарий нужен для сильного ансамбля.

То есть stats-модель хуже сама по себе, но ошибается по-другому. Для ансамбля это часто ценнее, чем ещё одна модель, которая на 0.03 лучше, но повторяет ошибки основной. Это как раз тот эффект, ради которого стоило проверить target statistics.

Сейчас не делаем ни OOF, ни новые группы, ни подбор smoothing. Сначала проверяем, что эффект не привязан к seed=42.

Критически важно: вес 0.45 уже выбран на seed=42, поэтому на следующих split не оптимизируем его заново. Просто фиксируем:

In [8]:
FIXED_BASELINE_WEIGHT = 0.55
FIXED_STATS_WEIGHT = 0.45

Ниже функция, которая повторяет ровно текущий эксперимент для одного внешнего seed. Она использует уже созданные у тебя:

X_model_v7
X_raw
y
categorical_columns_v7
GROUP_SPECS
make_cross_fitted_target_stats()
train_catboost_holdout()
mape_percent()
normalize_title_for_shift()

In [9]:
def run_target_stats_holdout(
    outer_seed: int,
    stats_weight: float = 0.45,
):
    price_bins = np.asarray(
        pd.qcut(
            y,
            q=10,
            labels=False,
            duplicates="drop",
        )
    )

    train_idx, valid_idx = train_test_split(
        np.arange(len(y)),
        test_size=0.20,
        random_state=outer_seed,
        stratify=price_bins,
    )

    X_train_base = X_model_v7.iloc[train_idx].copy()
    X_valid_base = X_model_v7.iloc[valid_idx].copy()

    y_train_outer = y.iloc[train_idx].copy()
    y_valid_outer = y.iloc[valid_idx].copy()

    X_train_stats, X_valid_stats = (
        make_cross_fitted_target_stats(
            X_fit=X_train_base,
            y_fit=y_train_outer,
            X_apply=X_valid_base,
            group_specs=GROUP_SPECS,
            smoothing=20.0,
            n_splits=5,
            random_state=142,
        )
    )

    X_train_with_stats = pd.concat(
        [X_train_base, X_train_stats],
        axis=1,
    )

    X_valid_with_stats = pd.concat(
        [X_valid_base, X_valid_stats],
        axis=1,
    )

    baseline_model, baseline_pred = train_catboost_holdout(
        X_train=X_train_base,
        y_train=y_train_outer,
        X_valid=X_valid_base,
        y_valid=y_valid_outer,
        categorical_columns=categorical_columns_v7,
    )

    stats_model, stats_pred = train_catboost_holdout(
        X_train=X_train_with_stats,
        y_train=y_train_outer,
        X_valid=X_valid_with_stats,
        y_valid=y_valid_outer,
        categorical_columns=categorical_columns_v7,
    )

    blend_pred = (
        (1 - stats_weight) * baseline_pred
        + stats_weight * stats_pred
    )

    title_train = normalize_title_for_shift(
        X_raw.iloc[train_idx]["Полное название"]
    )

    title_valid = normalize_title_for_shift(
        X_raw.iloc[valid_idx]["Полное название"]
    )

    unseen_title_mask = (
        ~title_valid.isin(set(title_train))
    ).to_numpy()

    low_price_threshold = y_train_outer.quantile(0.25)

    low_price_mask = (
        y_valid_outer.to_numpy() <= low_price_threshold
    )

    y_valid_array = y_valid_outer.to_numpy()

    result = {
        "seed": outer_seed,
        "baseline_best_iteration": (
            baseline_model.get_best_iteration()
        ),
        "stats_best_iteration": (
            stats_model.get_best_iteration()
        ),
        "baseline_mape_pct": mape_percent(
            y_valid_array,
            baseline_pred,
        ),
        "stats_mape_pct": mape_percent(
            y_valid_array,
            stats_pred,
        ),
        "blend_mape_pct": mape_percent(
            y_valid_array,
            blend_pred,
        ),
        "blend_delta_vs_baseline_pp": (
            mape_percent(y_valid_array, blend_pred)
            - mape_percent(y_valid_array, baseline_pred)
        ),
        "baseline_unseen_title_mape_pct": mape_percent(
            y_valid_array[unseen_title_mask],
            baseline_pred[unseen_title_mask],
        ),
        "blend_unseen_title_mape_pct": mape_percent(
            y_valid_array[unseen_title_mask],
            blend_pred[unseen_title_mask],
        ),
        "baseline_low_price_mape_pct": mape_percent(
            y_valid_array[low_price_mask],
            baseline_pred[low_price_mask],
        ),
        "blend_low_price_mape_pct": mape_percent(
            y_valid_array[low_price_mask],
            blend_pred[low_price_mask],
        ),
    }

    return result

In [10]:
seed_202_result = run_target_stats_holdout(
    outer_seed=202,
    stats_weight=0.45,
)

display(
    pd.DataFrame([seed_202_result])
)

0:	learn: 0.6520416	test: 0.6561286	best: 0.6561286 (0)	total: 77.4ms	remaining: 3m 52s
500:	learn: 0.1562917	test: 0.2033491	best: 0.2033491 (500)	total: 1m 4s	remaining: 5m 21s
1000:	learn: 0.1162002	test: 0.1923048	best: 0.1923048 (1000)	total: 1m 57s	remaining: 3m 54s
1500:	learn: 0.0941059	test: 0.1881030	best: 0.1881004 (1498)	total: 3m	remaining: 2m 59s
2000:	learn: 0.0786093	test: 0.1863390	best: 0.1863244 (1998)	total: 4m 2s	remaining: 2m 1s
2500:	learn: 0.0664220	test: 0.1854236	best: 0.1854112 (2493)	total: 5m 4s	remaining: 1m
2999:	learn: 0.0567549	test: 0.1849999	best: 0.1849977 (2970)	total: 5m 49s	remaining: 0us

bestTest = 0.1849977495
bestIteration = 2970

Shrink model to first 2971 iterations.
0:	learn: 0.6502115	test: 0.6542536	best: 0.6542536 (0)	total: 71.6ms	remaining: 3m 34s
500:	learn: 0.1419564	test: 0.1941895	best: 0.1941895 (500)	total: 52.5s	remaining: 4m 21s
1000:	learn: 0.1055685	test: 0.1842899	best: 0.1842899 (1000)	total: 1m 39s	remaining: 3m 18s
1500:	

,seed,baseline_best_iteration,stats_best_iteration,baseline_mape_pct,stats_mape_pct,blend_mape_pct,blend_delta_vs_baseline_pp,baseline_unseen_title_mape_pct,blend_unseen_title_mape_pct,baseline_low_price_mape_pct,blend_low_price_mape_pct
0,202,2970,2998,12.033115,11.611377,11.561568,-0.471547,15.442614,14.911588,16.224179,15.823386


In [11]:
seed_2026_result = run_target_stats_holdout(
    outer_seed=2026,
    stats_weight=0.45,
)

replication_results = pd.DataFrame(
    [
        {
            "seed": 42,
            "baseline_mape_pct": 12.806429,
            "stats_mape_pct": 12.868201,
            "blend_mape_pct": 12.566674,
            "blend_delta_vs_baseline_pp": -0.239755,
        },
        seed_202_result,
        seed_2026_result,
    ]
)

display(replication_results)

0:	learn: 0.6513553	test: 0.6545238	best: 0.6545238 (0)	total: 134ms	remaining: 6m 43s
500:	learn: 0.1486920	test: 0.2114989	best: 0.2114989 (500)	total: 53s	remaining: 4m 24s
1000:	learn: 0.1145179	test: 0.2027652	best: 0.2027652 (1000)	total: 1m 37s	remaining: 3m 15s
1500:	learn: 0.0917619	test: 0.2000460	best: 0.2000122 (1492)	total: 2m 23s	remaining: 2m 23s
2000:	learn: 0.0774918	test: 0.1992756	best: 0.1992675 (1958)	total: 3m 10s	remaining: 1m 34s
2500:	learn: 0.0657945	test: 0.1985595	best: 0.1985572 (2499)	total: 4m 2s	remaining: 48.5s
2999:	learn: 0.0566540	test: 0.1982266	best: 0.1982266 (2999)	total: 5m 7s	remaining: 0us

bestTest = 0.198226558
bestIteration = 2999

0:	learn: 0.6521431	test: 0.6552893	best: 0.6552893 (0)	total: 125ms	remaining: 6m 14s
500:	learn: 0.1356261	test: 0.2057044	best: 0.2057044 (500)	total: 56.9s	remaining: 4m 43s
1000:	learn: 0.1025261	test: 0.1992438	best: 0.1992438 (1000)	total: 1m 55s	remaining: 3m 50s
1500:	learn: 0.0812515	test: 0.1962767	bes

,seed,baseline_mape_pct,stats_mape_pct,blend_mape_pct,blend_delta_vs_baseline_pp,baseline_best_iteration,stats_best_iteration,baseline_unseen_title_mape_pct,blend_unseen_title_mape_pct,baseline_low_price_mape_pct,blend_low_price_mape_pct
0,42,12.806429,12.868201,12.566674,-0.239755,NaN,NaN,NaN,NaN,NaN,NaN
1,202,12.033115,11.611377,11.561568,-0.471547,2970.0,2998.0,15.442614,14.911588,16.224179,15.823386
2,2026,12.548894,12.492653,12.252789,-0.296105,2999.0,2994.0,16.219305,15.893581,17.146229,16.863308
